In [ ]:
import arcpy
import pandas as pd
import numpy as np

In [ ]:
# setting workspace
arcpy.env.workspace = "YOUR/PATH/TO/industrial-pollution-risk.gdb"
arcpy.env.overwriteOutput = True

## Join census and air pollution data
Joining these two datasets allows me to model air pollution and demographic characteristics more generally, not related to schools.

In [ ]:
# joining census demographic data with rsei data
arcpy.management.AddJoin(
    in_layer_or_view="cens_demg",
    in_field="GEOID",
    join_table="rsei_cens",
    join_field="GEOID",
    join_type="KEEP_ALL",
    index_join_fields="NO_INDEX_JOIN_FIELDS",
    rebuild_index="NO_REBUILD_INDEX",
    join_operation=""
)

## Projections for distance calculations
In order to calculate geodesic distances in kilometers, I have to project the data using a projected coordinate system (in this case, the [Equi7 North America](https://epsg.io/27705) projection).

In [ ]:
# converting schools and sites data to projected coordinate system for calculating geodesic distance
arcpy.management.Project(
    in_dataset="SCH_XY", 
    out_dataset="SCH", 
    out_coor_system=arcpy.SpatialReference(27705)
)
arcpy.management.Project(
    in_dataset="TRI_XY", 
    out_dataset="TRI", 
    out_coor_system=arcpy.SpatialReference(27705)
)

## Join school data with underlying pollution data
The school data (points) can't be joined directly with the pollution or census data (polygons); instead, I spatially join the two to pull underlying pollution and census data at the location of each school. These data reflect averages across each census tract, not the actual enrollment of each school or the pollution at the exact location of each school.

In [ ]:
# joining school data to underlying rsei data
arcpy.analysis.SpatialJoin(
    target_features="SCH",
    join_features="rsei_cens Join_Count",
    out_feature_class="SCH_rsei",
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    field_mapping=None,
    match_option="INTERSECT",
    search_radius=None,
    distance_field_name="",
    match_fields=None
)

# joining school data to census data
arcpy.analysis.SpatialJoin(
    target_features="SCH_rsei",
    join_features="cens_demg Join_Count",
    out_feature_class="SCH_rsei_cens",
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="INTERSECT",
    search_radius=None,
    distance_field_name="",
    match_fields=None
)

## Subset school data into proximity groups
This data is useful for modeling differences between the two proximity groups. I chose 5 km as the cutoff based on air pollution research; studies often define proximity cutoffs at 1, 2, 5, or 10 km, so 5 km represents a middle ground between extreme exposure and minimal impact.

In [ ]:
# creating 5 km buffers around toxic sites
arcpy.analysis.Buffer(
    in_features="TRI",
    out_feature_class="TRI_5kmB",
    buffer_distance_or_field="5 Kilometers",
    line_side="FULL",
    line_end_type="ROUND",
    dissolve_option="NONE",
    dissolve_field=None,
    method="GEODESIC"
)

# getting subset of public schools within 5 km of toxic sites
arcpy.analysis.Clip(
    in_features="SCH_rsei_cens",
    clip_features="TRI_5kmB",
    out_feature_class="SCH_5km",
    cluster_tolerance=None
)

# getting subset of public schools outside of 5 km of toxic sites
arcpy.analysis.Erase(
    in_features="SCH_rsei_cens",
    erase_features="TRI_5kmB",
    out_feature_class="SCH_over5km",
    cluster_tolerance=None
)

## Subset polluter data into proximity groups

In [ ]:
# 5 km buffers around schools
arcpy.analysis.Buffer(
    in_features="SCH_rsei_cens",
    out_feature_class="SCH_5kmB",
    buffer_distance_or_field="5 Kilometers",
    line_side="FULL",
    line_end_type="ROUND",
    dissolve_option="NONE",
    dissolve_field=None,
    method="GEODESIC"
)

# getting toxic sites within 5 km of public schools
arcpy.analysis.Clip(
    in_features="TRI",
    clip_features="SCH_5kmB",
    out_feature_class="TRI_5km",
    cluster_tolerance=None
)

## Calculate exact distances from schools to nearest site
In addition to proximity groups above, the exact distances will also be useful for modeling.

In [ ]:
# calculating distance in km to nearest industrial site
arcpy.analysis.Near(
    in_features="SCH",
    near_features="TRI",
    search_radius=None,
    location="NO_LOCATION",
    angle="NO_ANGLE",
    method="GEODESIC",
    field_names="NEAR_FID NEAR_FID #;NEAR_DIST NEAR_DIST #",
    distance_unit="Kilometers",
    match_fields=None
)

## Join with HOLC data
The proximity data also needs to be spatially joined to the underlying HOLC grades for modeling.

In [ ]:
# spatial join to get underlying HOLC grades associated with each school
arcpy.analysis.SpatialJoin(
    target_features="SCH",
    join_features="HOLC",
    out_feature_class="SCH_HOLC_all",
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_COMMON",
    field_mapping=None,
    match_option="INTERSECT",
    search_radius=None,
    distance_field_name="",
    match_fields=None
)

# select and export only the regions with HOLC grades or regions noted as industrial zones
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view="SCH_HOLC_all", 
    selection_type="NEW_SELECTION",
    where_clause="grade IS NOT NULL Or industrial IS NOT NULL"
)
arcpy.management.CopyFeatures(
    in_features="SCH_HOLC_all", 
    out_feature_class="SCH_HOLC"
)